# Agent Roadwork (Phase 4)

Publishes roadwork closure events for fixed segment IDs using configured tick windows.

In [5]:
# Cell purpose: Load config, connect MQTT, and prepare the roadwork topic.
from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.topics import roadwork_events_topic

config = load_config()
phase1 = config.simulation.car_rerouting_phase1 if config.simulation else None
if phase1 is None:
    raise ValueError("Missing simulation.car_rerouting_phase1 in config.yaml")

client = mqtt.connect_mqtt(config.mqtt, client_id_suffix="agent-roadwork-phase4")
topic = roadwork_events_topic(config.mqtt.base_topic)

print(f"Connected MQTT target: {config.mqtt.host}:{config.mqtt.port}")
print(f"Roadwork topic: {topic}")
print(
    f"Roadwork window: start_tick={phase1.roadwork.start_tick}, "
    f"end_tick={phase1.roadwork.end_tick}, blocked={list(phase1.roadwork.blocked_segment_ids)}"
)

Connected MQTT target: 127.0.0.1:1883
Roadwork topic: simulated-city/city/roadwork/events
Roadwork window: start_tick=180, end_tick=300, blocked=[44105317, 733901267]


In [6]:
# Cell purpose: Publish scheduled roadwork closure/open events for selected ticks.
from datetime import datetime, timedelta, timezone

base_time = datetime(2026, 1, 1, tzinfo=timezone.utc)
scheduled_ticks = [1, phase1.roadwork.start_tick, phase1.roadwork.end_tick, phase1.max_ticks]
published = 0

try:
    for tick in scheduled_ticks:
        is_active = phase1.roadwork.start_tick <= tick <= phase1.roadwork.end_tick
        payload = {
            "agent": "agent_roadwork",
            "tick": tick,
            "timestamp": (base_time + timedelta(seconds=tick * phase1.tick_seconds)).isoformat(),
            "active": is_active,
            "blocked_segment_ids": list(phase1.roadwork.blocked_segment_ids),
        }
        mqtt.publish_json_checked(client, topic, payload)
        published += 1

    print(f"Published roadwork events: {published}")
finally:
    client.loop_stop()
    client.disconnect()
    print("Disconnected MQTT client.")

Published roadwork events: 4
Disconnected MQTT client.
